# Week 7 · The aggregate planning lab: three plans for the roofing supplier

**Sentence of the day:** *Demand goes up and down; a factory would rather not. An aggregate plan decides who absorbs the swing — the warehouse (stock), a partner (subcontracting) or the people (hiring and layoffs) — and every choice has its price.*

This lab rebuilds the worked example of Heizer, Render & Munson, Chapter 13 (Examples 1–4): a roofing-supplies maker in Juárez, Mexico, plans January to June. You can check every number against the book.

**What you will do (about 60 minutes)**

| Part | What | Time | You type |
|---|---|---|---|
| 0 | Setup | 2 min | nothing |
| 1 | ✏️ The problem: demand per day against the average (Figure 13.3) | 10 min | one blank |
| 2 | ✏️ Plan 1 · level: constant workforce, stock absorbs the swing | 12 min | one blank |
| 3 | ✏️ Plan 2 · level low + subcontracting | 8 min | one blank |
| 4 | ✏️ Plan 3 · chase: hire and lay off | 10 min | one blank |
| 5 | ▶ The three plans side by side (Table 13.5) | 5 min | nothing |
| 6 | ▶ Play: when does the winner change? | 10 min | nothing – you play |
| 7 | ⭐ Extra: the same plans priced in CO₂ | if time | nothing |

**How this notebook works**

- Google Colab is a notebook: a document where text and runnable code sit side by side. You run a code cell with the ▶ button on its left, or with **Shift + Enter**.
- Run the cells **from top to bottom**. A cell can only use what the cells above it have created.
- Lines that start with `#` are **comments**: notes in plain English for you. Python skips them.
- A cell marked **✏️ YOUR TURN** has one blank `___`. Replace it, run the cell, then run the **CHECK** cell below it: ✅ = right, ❌ = look at the hint.
- A **slider** is a little bar you drag with the mouse. The chart redraws each time you let go.
- First save your own copy: **File → Save a copy in Drive**.

## Part 0 · Setup

This cell loads the **libraries** (toolboxes other people have written): `pandas` for tables, `numpy` for numbers, `matplotlib` for charts, `ipywidgets` for sliders. All are installed in Colab. It also defines `check`, the helper that prints ✅ or ❌. Run it once.

In [ ]:
# ── Part 0 · Setup ──────────────────────────────────────────────────────────────
# A line that starts with # is a COMMENT: a note for humans. Python skips it.
import pandas as pd               # tables with named columns ("DataFrames"), like a spreadsheet
import numpy as np                # lists of numbers
import matplotlib.pyplot as plt   # drawing charts
from ipywidgets import interact, IntSlider, FloatSlider   # sliders

# Our colours (hex codes: #rrggbb). Names in CAPITALS = set once, never changed.
GREY, RED, DARK, GREEN, LIGHT, PINK = "#a6a6a6", "#c1002a", "#1a1a1a", "#2e7d4f", "#f2f2f2", "#e07a8c"
PLAN_COLOURS = {"Plan 1 · level": DARK, "Plan 2 · subcontract": GREY, "Plan 3 · chase": RED}

# One chart style for the whole notebook.
plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white", "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})

# (Technical – you may skip this.) Make an unfilled ___ stop the cell with a clear message.
try:
    get_ipython().user_ns.pop("___", None)
except NameError:
    pass


def check(label, got, expected, tol=0.01, hint=""):
    # Compare your result with the book's number and print ✅ or ❌.
    try:
        ok = abs(float(got) - float(expected)) <= tol      # True when the two are close enough
        shown = f"{float(got):,.2f}"
    except Exception:
        ok, shown = False, str(got)
    print(("✅ " if ok else "❌ ") + f"{label}: you have {shown}, expected {expected:,.2f}"
          + ("" if ok else f"   Hint: {hint}"))


def not_yet(error):
    # Turns Python's error message into a plain hint – usually: a ___ blank is still waiting.
    if "___" in str(error) or isinstance(error, NameError):
        print("❌ Not done yet: the cell above has not run to its end. Replace the ___ there, run it, then run this check again.")
    else:
        print("❌ Not working yet:", error)


print("Setup done.")

## Part 1 · ✏️ The problem: demand goes up and down, the factory would rather not

The roofing supplier has a **forecast** for one product family (all roofing products added up — that is what *aggregate* means: we plan the whole family in one number, not each product). The months have different numbers of **production days**, so we compare months by their **demand per day**.

**Picture it like a bathtub:** production is the tap, demand is the drain, inventory is the water in the tub. A steady tap with a drain that changes speed means the water level rises and falls.

Run the cell: it shows Table 13.2 from the book.

In [ ]:
# ── 1 · Table 13.2: monthly forecasts ─────────────────────────────────────────────
plan = pd.DataFrame({
    "month":  ["Jan", "Feb", "Mar", "Apr", "May", "Jun"],
    "demand": [900, 700, 800, 1200, 1500, 1100],     # expected demand in units (forecast)
    "days":   [22, 18, 21, 21, 22, 20],              # production (working) days in the month
}).set_index("month")                                 # the month names become the row labels

# Demand per day = demand ÷ days, rounded to whole units as in the book
plan["demand per day"] = (plan["demand"] / plan["days"]).round().astype(int)

print("Total demand:", plan["demand"].sum(), "units   ·   total production days:", plan["days"].sum())
plan

**✏️ YOUR TURN:** a factory that wants to produce at one steady pace needs the **average requirement**:

**average requirement = total expected demand ÷ total number of production days**

Replace the `___` with the total number of production days. (Hint: look at how the total demand is written in the line above it.)

In [ ]:
# ── 1 · The average requirement and Figure 13.3 ───────────────────────────────────
total_demand = plan["demand"].sum()           # 6,200 units
# ✏️ YOUR TURN: divide by the TOTAL number of production days
average_per_day = total_demand / ___
print(f"Average requirement: {average_per_day:.0f} units per day")

# Picture: demand per day as steps (each step as wide as the month has days), the average as a dashed line
fig, ax = plt.subplots(figsize=(10, 4.2))
edges = np.concatenate([[0], plan["days"].cumsum().values])     # where each month starts and ends on the day axis
ax.stairs(plan["demand per day"], edges, fill=True, color=PINK, alpha=0.35)
ax.stairs(plan["demand per day"], edges, color=RED, lw=2.5, label="Forecast demand per day")
ax.axhline(average_per_day, color=DARK, lw=2, ls="--", label=f"Level production: {average_per_day:.0f} per day")
mids = (edges[:-1] + edges[1:]) / 2
ax.set_xticks(mids, [f"{m}\n{d} days" for m, d in zip(plan.index, plan["days"])])
for x, y in zip(mids, plan["demand per day"]):
    ax.text(x, y + 1.5, y, ha="center", fontweight="bold")
ax.set(ylim=(0, 80), ylabel="units per working day", title="Figure 13.3 · Demand per day against the average")
ax.legend(frameon=False, loc="upper left"); plt.tight_layout(); plt.show()

In [ ]:
# CHECK 1
try:
    check("average requirement", average_per_day, 50, hint="average_per_day = total_demand / plan['days'].sum()")
except Exception as e:
    not_yet(e)

**Read the picture:** January to March lie *below* the dashed line (the steady factory would make too much), April to June lie *above* it (too little). Every aggregate plan is an answer to one question: **who absorbs the gap between the red steps and the dashed line?**

The book prices three answers with the costs of Table 13.3:

In [ ]:
# ── Table 13.3: cost information (the book's numbers, in US dollars) ──────────────────
HOLD      = 5      # $ per unit kept in stock for one month (inventory carrying cost)
SUBCON    = 20     # $ per unit bought from a subcontractor
PAY_DAY   = 80     # $ per worker per 8-hour day ($10 an hour)
HOURS     = 1.6    # labour-hours to make one unit
HIRE      = 300    # $ per unit of INCREASE in the daily production rate (hiring and training)
LAYOFF    = 600    # $ per unit of DECREASE in the daily production rate (layoffs)

UNITS_PER_WORKER = 8 / HOURS          # one worker makes 8 h ÷ 1.6 h = 5 units a day
print(f"One worker makes {UNITS_PER_WORKER:.0f} units a day.")

## Part 2 · ✏️ Plan 1 · level: a constant workforce, the warehouse absorbs the swing

The factory makes **50 units every day**, with no overtime, no subcontracting, no hiring. In the slow months the tub fills (inventory grows), in the busy months it drains. Start and end inventory = 0.

- Workers needed: 50 units a day ÷ 5 units per worker = **10 workers**.
- **Inventory carried** = the ending inventories of the six months added up (the book's simple rule: each unit left at the end of a month is charged one month of carrying cost).

In [ ]:
# ── 2 · Plan 1 month by month (the table of Example 2) ──────────────────────────────
RATE_1 = 50                                     # units a day, every day
p1 = pd.DataFrame(index=plan.index)
p1["production"] = RATE_1 * plan["days"]        # 50 × days
p1["demand"] = plan["demand"]
p1["change"] = p1["production"] - p1["demand"]  # + = the tub fills, − = it drains
p1["ending inventory"] = p1["change"].cumsum()  # cumsum = running total: Jan, Jan+Feb, Jan+Feb+Mar, …
workers_1 = RATE_1 / UNITS_PER_WORKER

print(f"Workers: {workers_1:.0f}   ·   inventory carried: {p1['ending inventory'].sum():,} units")
p1

Two pictures of the same plan. **Left:** each month, the steady bar of production against the demand that moves, and the inventory left at the month's end. **Right:** the *cumulative* view — everything made so far against everything sold so far. The vertical gap between the two lines is the water in the tub.

In [ ]:
# ── 2 · Two pictures of plan 1 ─────────────────────────────────────────────────────
fig, (left, right) = plt.subplots(1, 2, figsize=(13, 4.3))
x = np.arange(6)
left.bar(x - 0.2, p1["production"], 0.4, color=DARK, label="production (50 a day)")
left.bar(x + 0.2, p1["demand"], 0.4, color=RED, label="demand")
left.plot(x, p1["ending inventory"], color=GREEN, lw=3, marker="o", label="ending inventory")
left.set_xticks(x, plan.index); left.set(title="Plan 1 month by month", ylabel="units")
left.legend(frameon=False, fontsize=9)

cum_made, cum_sold = p1["production"].cumsum(), p1["demand"].cumsum()
right.plot(x, cum_made, color=DARK, lw=3, marker="o", label="made so far")
right.plot(x, cum_sold, color=RED, lw=3, marker="o", label="sold so far")
right.fill_between(x, cum_sold, cum_made, color=GREEN, alpha=0.25, label="the gap = stock")
right.annotate(f"{p1['ending inventory'].max()} units\nin stock", xy=(2, (cum_made.iloc[2] + cum_sold.iloc[2]) / 2),
               xytext=(0.2, 4200), arrowprops=dict(arrowstyle="->"), fontsize=10)
right.set_xticks(x, plan.index); right.set(title="Plan 1 cumulative: made vs sold", ylabel="units (running total)")
right.legend(frameon=False, fontsize=9, loc="lower right")
plt.tight_layout(); plt.show()

**✏️ YOUR TURN:** price plan 1.

- inventory cost = units carried × carrying cost per unit and month
- regular labour = workers × pay per day × total production days

Replace the `___` with the name of the carrying cost from Table 13.3 (a name written in CAPITALS above).

In [ ]:
# ── 2 · The cost of plan 1 ──────────────────────────────────────────────────────────
carried_1 = p1["ending inventory"].sum()                 # 1,850 units carried
# ✏️ YOUR TURN: units carried × carrying cost per unit and month
inventory_cost_1 = carried_1 * ___
labour_1 = workers_1 * PAY_DAY * plan["days"].sum()      # 10 × $80 × 124 days
total_1 = inventory_cost_1 + labour_1
print(f"Inventory carrying: ${inventory_cost_1:,.0f}")
print(f"Regular labour:     ${labour_1:,.0f}")
print(f"Total plan 1:       ${total_1:,.0f}")

In [ ]:
# CHECK 2
try:
    check("inventory cost plan 1", inventory_cost_1, 9250, hint="carried_1 × HOLD (1,850 × $5)")
    check("total plan 1", total_1, 108450)
except Exception as e:
    not_yet(e)

## Part 3 · ✏️ Plan 2 · level low, a partner absorbs the swing

Again a constant workforce, but set so low that it only covers the **slowest month** (March, 800 ÷ 21 = 38 units a day). Everything above that is bought from a **subcontractor** at $20 a unit. No stock is built, so there is no carrying cost.

- Workers: 38 ÷ 5 = **7.6 workers** (think: 7 full-time and one part-timer).

In [ ]:
# ── 3 · Plan 2: in-house at the slowest month's pace, the rest bought in ───────────────
RATE_2 = plan["demand per day"].min()              # 38 units a day (March)
workers_2 = RATE_2 / UNITS_PER_WORKER              # 7.6 workers

p2 = pd.DataFrame(index=plan.index)
p2["in-house"] = RATE_2 * plan["days"]             # what our own people make each month
p2["subcontracted"] = plan["demand"] - p2["in-house"]   # what we buy to fill the gap
p2["demand"] = plan["demand"]
print(f"Workers: {workers_2:.1f}   ·   in-house: {p2['in-house'].sum():,}   ·   subcontracted: {p2['subcontracted'].sum():,} units")

# Picture: a stacked bar per month — our own production at the bottom, the partner's on top
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(plan.index, p2["in-house"], color=DARK, label=f"in-house ({RATE_2} a day)")
ax.bar(plan.index, p2["subcontracted"], bottom=p2["in-house"], color=GREY, label="subcontracted")
ax.set(title="Plan 2: the subcontractor fills every month's gap", ylabel="units")
ax.legend(frameon=False); plt.tight_layout(); plt.show()
p2

**✏️ YOUR TURN:** subcontracting cost = units subcontracted × price per subcontracted unit. Replace the `___` with the right name from Table 13.3.

In [ ]:
# ── 3 · The cost of plan 2 ──────────────────────────────────────────────────────────
sub_units_2 = p2["subcontracted"].sum()                   # 1,488 units
labour_2 = workers_2 * PAY_DAY * plan["days"].sum()       # 7.6 × $80 × 124 days
# ✏️ YOUR TURN: units subcontracted × price per subcontracted unit
subcon_cost_2 = sub_units_2 * ___
total_2 = labour_2 + subcon_cost_2
print(f"Regular labour:  ${labour_2:,.0f}")
print(f"Subcontracting:  ${subcon_cost_2:,.0f}")
print(f"Total plan 2:    ${total_2:,.0f}")

In [ ]:
# CHECK 3
try:
    check("subcontracting cost plan 2", subcon_cost_2, 29760, hint="sub_units_2 × SUBCON (1,488 × $20)")
    check("total plan 2", total_2, 105152)
except Exception as e:
    not_yet(e)

## Part 4 · ✏️ Plan 3 · chase: the workforce absorbs the swing

Now production **chases** demand: each month the factory makes exactly that month's demand. No stock, no subcontractor — instead the daily rate (and with it the number of people) goes up and down.

- Raising the daily rate by one unit costs **$300** (hiring and training).
- Lowering it by one unit costs **$600** (layoffs).
- December's rate was the same as January's (41), so January has no change.
- Basic production cost = demand × 1.6 hours × $10.

In [ ]:
# ── 4 · Plan 3: the daily rate follows demand (Table 13.4) ──────────────────────────────
p3 = pd.DataFrame(index=plan.index)
p3["demand"] = plan["demand"]
p3["daily rate"] = plan["demand per day"]
p3["change"] = p3["daily rate"].diff().fillna(0).astype(int)   # diff = this month − last month (Jan: 0)
p3["increase"] = p3["change"].clip(lower=0)                    # keep only the ups (downs become 0)
p3["decrease"] = (-p3["change"]).clip(lower=0)                 # keep only the downs, as positive numbers
p3["basic cost"] = p3["demand"] * HOURS * 10                   # 1.6 h × $10 an hour per unit
p3["hiring cost"] = p3["increase"] * HIRE

# Picture: the daily rate month by month; green arrows = hiring, red arrows = layoffs
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(plan.index, p3["daily rate"], color=LIGHT, edgecolor=DARK)
for i, (m, row) in enumerate(p3.iterrows()):
    if row["change"] != 0:
        ax.annotate("", xy=(i, row["daily rate"]), xytext=(i, row["daily rate"] - row["change"]),
                    arrowprops=dict(arrowstyle="-|>", lw=3, color=GREEN if row["change"] > 0 else RED))
        ax.text(i + 0.12, row["daily rate"] - row["change"] / 2, f"{int(row['change']):+d}",
                color=GREEN if row["change"] > 0 else RED, fontweight="bold")
ax.set(title="Plan 3: the daily rate chases demand (green = hire, red = lay off)", ylabel="units per day", ylim=(0, 80))
plt.tight_layout(); plt.show()
p3

**✏️ YOUR TURN:** the layoff cost is still missing. Layoff cost = decrease in the daily rate × cost per unit of decrease. Replace the `___`.

In [ ]:
# ── 4 · The cost of plan 3 ──────────────────────────────────────────────────────────
# ✏️ YOUR TURN: decrease in the daily rate × cost per unit of decrease (Table 13.3)
p3["layoff cost"] = p3["decrease"] * ___
p3["total"] = p3["basic cost"] + p3["hiring cost"] + p3["layoff cost"]

labour_3, hiring_3, layoff_3 = p3["basic cost"].sum(), p3["hiring cost"].sum(), p3["layoff cost"].sum()
total_3 = p3["total"].sum()
print(f"Basic production: ${labour_3:,.0f}")
print(f"Hiring:           ${hiring_3:,.0f}")
print(f"Layoffs:          ${layoff_3:,.0f}")
print(f"Total plan 3:     ${total_3:,.0f}")
p3[["demand", "daily rate", "basic cost", "hiring cost", "layoff cost", "total"]]

In [ ]:
# CHECK 4
try:
    check("layoff cost plan 3", layoff_3, 9600, hint="p3['decrease'] × LAYOFF (16 units of decrease × $600)")
    check("total plan 3", total_3, 117800)
except Exception as e:
    not_yet(e)

## Part 5 · ▶ The three plans side by side (Table 13.5)

Run the cell. Each bar is one plan, cut into its cost blocks. The shortest bar is the cheapest plan — under *these* cost numbers.

In [ ]:
# ── 5 · Table 13.5 as a table and as a stacked bar ────────────────────────────────────
compare = pd.DataFrame({
    "Plan 1 · level":       {"inventory": inventory_cost_1, "regular labour": labour_1, "hiring": 0, "layoffs": 0, "subcontracting": 0},
    "Plan 2 · subcontract": {"inventory": 0, "regular labour": labour_2, "hiring": 0, "layoffs": 0, "subcontracting": subcon_cost_2},
    "Plan 3 · chase":       {"inventory": 0, "regular labour": labour_3, "hiring": hiring_3, "layoffs": layoff_3, "subcontracting": 0},
})
compare.loc["TOTAL"] = compare.sum()

BLOCK_COLOURS = {"regular labour": LIGHT, "inventory": GREEN, "subcontracting": GREY, "hiring": PINK, "layoffs": RED}
fig, ax = plt.subplots(figsize=(10, 3.6))
left = np.zeros(3)
for block, colour in BLOCK_COLOURS.items():
    ax.barh(compare.columns, compare.loc[block], left=left, color=colour, edgecolor=DARK, label=block)
    left += compare.loc[block].values
for y, total in enumerate(left):
    ax.text(total + 800, y, f"${total:,.0f}", va="center", fontweight="bold")
ax.set_xlim(0, 135000); ax.invert_yaxis()
ax.set(title=f"Cheapest: {compare.loc['TOTAL'].idxmin()}", xlabel="$ for January–June")
ax.legend(frameon=False, ncol=5, loc="lower center", bbox_to_anchor=(0.5, -0.42))
plt.show()
compare.style.format("${:,.0f}")

**Read the picture:** all three plans pay about the same for making the units (the light block). They differ in **who absorbs the swing**: the warehouse (green), the subcontractor (grey) or the workforce (pink and red). Plan 2 wins here — by about $3,300 over plan 1. That is a thin margin: the next part shows how easily it flips.

## Part 6 · ▶ Play: when does the winner change?

The cell below puts all three plans into one function and gives you four sliders. Nothing to type — drag and watch which bar is shortest.

**Three questions to answer with the sliders** (write the answer on paper, then compare with your neighbour):

1. Set June demand to **1,200** (the book's Learning Exercises). Which plan's cost moves the most, and why? *(Book: plan 2 → $107,152; plan 3 → $116,400.)*
2. Put June back to 1,100. Raise the **subcontracting price**. Above roughly what price does plan 1 beat plan 2?
3. Back to $20. Lower the **carrying cost**. Below roughly what cost does plan 1 win?

*Tip for the AI check:* ask an AI the same question 2 and let it calculate the break-even price. Does its answer match your slider? Where does it differ, and who is right?

In [ ]:
# ── 6 · All three plans as one function, with sliders ─────────────────────────────────
def three_plans(june=1100, hold=HOLD, subcon=SUBCON, layoff=LAYOFF):
    # Recalculate the three plans for a given June demand and three cost numbers. Returns $ totals.
    d = plan["demand"].copy(); d["Jun"] = june              # the forecast with the new June
    days = plan["days"]
    # Plan 1: 50 a day as in the book; if stock would go below 0, the missing units are subcontracted
    stock, carried, short = 0, 0, 0
    for made, sold in zip(RATE_1 * days, d):
        stock += made - sold
        if stock < 0:
            short += -stock; stock = 0
        carried += stock
    p1_cost = carried * hold + workers_1 * PAY_DAY * days.sum() + short * subcon
    # Plan 2: in-house at the slowest month's daily demand, the rest subcontracted
    rate = (d / days).round().astype(int)
    inhouse = rate.min() * days.sum()
    p2_cost = rate.min() / UNITS_PER_WORKER * PAY_DAY * days.sum() + (d.sum() - inhouse) * subcon
    # Plan 3: the daily rate chases demand
    change = rate.diff().fillna(0)
    p3_cost = d.sum() * HOURS * 10 + change.clip(lower=0).sum() * HIRE + (-change).clip(lower=0).sum() * layoff
    return pd.Series({"Plan 1 · level": p1_cost, "Plan 2 · subcontract": p2_cost, "Plan 3 · chase": p3_cost})


def show_plans(june=1100, carrying_cost=5.0, subcontract_price=20.0, layoff_cost=600):
    totals = three_plans(june, carrying_cost, subcontract_price, layoff_cost)
    fig, ax = plt.subplots(figsize=(10, 3.2))
    ax.barh(totals.index, totals, color=[PLAN_COLOURS[p] for p in totals.index])
    for y, v in enumerate(totals):
        ax.text(v + 800, y, f"${v:,.0f}", va="center", fontweight="bold")
    ax.set_xlim(0, 160000); ax.invert_yaxis()
    ax.set(title=f"Cheapest: {totals.idxmin()}", xlabel="$ for January–June")
    plt.show()


# The sliders. Each one starts at the book's value.
interact(show_plans,
         june=IntSlider(value=1100, min=800, max=1600, step=50, description="June demand"),
         carrying_cost=FloatSlider(value=5, min=0, max=10, step=0.25, description="carry $/unit·mo"),
         subcontract_price=FloatSlider(value=20, min=10, max=40, step=0.5, description="subcon $/unit"),
         layoff_cost=IntSlider(value=600, min=0, max=1200, step=50, description="layoff $/unit"));

In [ ]:
# CHECK 6 – the function must reproduce the book (Table 13.5 and the Learning Exercises)
try:
    book = three_plans()
    check("plan 1, book values", book["Plan 1 · level"], 108450)
    check("plan 2, book values", book["Plan 2 · subcontract"], 105152)
    check("plan 3, book values", book["Plan 3 · chase"], 117800)
    check("plan 3, June = 1,200", three_plans(june=1200)["Plan 3 · chase"], 116400)
except Exception as e:
    not_yet(e)

<details><summary><b>Answers to the three questions</b> (open only after you tried)</summary>

1. All three move. Plan 1's steady 50 a day now runs 100 units short in June; this model buys them from the subcontractor (+$2,000 → $110,450). Plan 2 buys the same 100 units (+$2,000 → $107,152). Plan 3 gets *cheaper* although demand rises ($116,400): June now drops by only 8 units a day from May instead of 13, and the $3,000 saved in layoffs outweighs the $1,600 for making 100 more units.
2. About **$22.20**: plan 2 = $75,392 + 1,488 × price; plan 1 = $108,450. They are equal at (108,450 − 75,392) ÷ 1,488 ≈ $22.22.
3. About **$3.20**: plan 1 = $99,200 + 1,850 × cost = $105,152 at cost ≈ $3.22.

The lesson: with a margin of 3 %, the "best" plan depends on cost numbers that are themselves estimates. A planner reports the break-even points, not only the winner.
</details>

## Part 7 · ⭐ Extra: the same plans priced in CO₂

This is a sustainability module, so price the plans a second time — in **kg CO₂e** (CO₂ equivalents: all greenhouse gases converted into the warming effect of CO₂). The two factors below are **teaching values** (our own assumptions, not from the book): a stored unit uses warehouse heating and light; a subcontracted unit travels 300 km by truck and is made in a less efficient plant.

Run the cell, then think: plan 3 emits nothing extra — but who pays its price?

In [ ]:
# ── 7 · Teaching values (assumptions!) ──────────────────────────────────────────────────
CO2_STORE = 0.4     # kg CO₂e per unit kept in stock for one month
CO2_SUBCON = 1.5    # kg CO₂e per unit made by the subcontractor and trucked in
CO2_PRICE = 100     # $ per tonne of CO₂e (the price we also use in the group project)

co2 = pd.Series({"Plan 1 · level": carried_1 * CO2_STORE,
                 "Plan 2 · subcontract": sub_units_2 * CO2_SUBCON,
                 "Plan 3 · chase": 0.0})
dollars = compare.loc["TOTAL"]

fig, (a, b) = plt.subplots(1, 2, figsize=(12, 3.4))
a.barh(dollars.index, dollars, color=[PLAN_COLOURS[p] for p in dollars.index]); a.invert_yaxis()
a.set(title="Cost in $", xlim=(0, 135000))
b.barh(co2.index, co2, color=[PLAN_COLOURS[p] for p in co2.index]); b.invert_yaxis()
b.set(title="Extra emissions in kg CO₂e", xlim=(0, 2600)); b.set_yticks([])
for y, v in enumerate(co2):
    b.text(v + 30, y, f"{v:,.0f} kg", va="center", fontweight="bold")
plt.tight_layout(); plt.show()

with_co2 = dollars + co2 / 1000 * CO2_PRICE
print("Cost including a CO₂ price of $100 a tonne:")
print(with_co2.round(0).to_string())

**Think it through:** at $100 a tonne the CO₂ price adds only a few hundred dollars and plan 2 still wins in dollars, while it emits the most. Plan 3 is the cleanest in CO₂, yet its layoffs carry a social cost that no column of Table 13.5 shows. "Sustainable" has three sides — profit, planet, people — and the three plans rank differently on each.

---

**Three sentences for your notes** (in your own words): what an aggregate plan decides; the three ways to absorb a demand swing and what each one costs; why the cheapest plan can flip.

---
*Transparency note: Bei der Erstellung dieses Notebooks wurde Claude (Anthropic) zur Programmierung, für Visualisierungen und als Reviewer (Rechen- und Konsistenzchecks) genutzt. Alle Inhalte wurden von Roman Bartnik geprüft, überarbeitet und verantwortet.*

**Source:** Heizer, J., Render, B., & Munson, C. (2023). *Operations management: Sustainability and supply chain management* (14th ed., Global ed.). Pearson. — Chapter 13, Examples 1–4, Tables 13.2–13.5. CO₂ factors in Part 7 are teaching values (own assumptions).